[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llm-engineering-certified/notebooks/day-14-rag-chatbot-capstone.ipynb#scrollTo=10a2b3c4)

---
# Day 14 · Capstone — Production RAG Chatbot with Memory and Gradio UI
**certified-journeys / llm-engineering-certified** · Day 14 · Capstone

> **Goal for today:** Build a production-ready multi-document RAG chatbot that ingests PDFs and web pages, stores embeddings in a persistent Chroma vector store, retrieves context with MMR re-ranking, streams answers through a conversational chain with ConversationSummaryMemory, and exposes the full pipeline as a Gradio interface.


## Capstone overview

This is the culminating project for **LLM Engineering with LangChain**. Every pattern you
learned across 14 days converges here:

```
PDFs + Web pages
  → PyPDFLoader / WebBaseLoader
  → RecursiveCharacterTextSplitter
  → OpenAIEmbeddings → Chroma (persistent)
  → MMR retriever (k=5)
  → ConversationSummaryMemory
  → ConversationalRetrievalChain
  → .stream()  ←── tokens flow here
  → Gradio ChatInterface  ←── user sees it here
```

**What makes this "production-ready":**
- Persistent vector store — survives restarts, no re-ingestion required
- MMR re-ranking — diverse context, not just the top duplicate chunks
- Summary memory — handles long conversations without blowing the context window
- Streaming — tokens appear as they're generated, not all at once
- Source citations — every answer shows which documents were used


In [ ]:
%pip install -q langchain langchain-openai langchain-community langchain-chroma \
    chromadb openai tiktoken pypdf gradio requests beautifulsoup4 lxml

## Step 1 · Configure environment and imports

We centralise all configuration here so every downstream cell can reference a single source
of truth. The `PERSIST_DIR` controls where Chroma stores its SQLite database — this is what
makes the vector store persistent across sessions.

| Config key | Value | Why |
|------------|-------|-----|
| `PERSIST_DIR` | `./chroma_capstone` | Chroma writes here; survives kernel restarts |
| `COLLECTION_NAME` | `capstone_docs` | Namespace within the Chroma database |
| `CHUNK_SIZE` | 1000 | Chars per chunk — balances context density vs. retrieval precision |
| `CHUNK_OVERLAP` | 200 | Prevents facts split across chunk boundaries |
| `MMR_K` | 5 | Number of diverse chunks returned per query |
| `MMR_FETCH_K` | 20 | Candidates fetched before MMR re-ranks them |


In [ ]:
import os
import json
from pathlib import Path

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain.memory import ConversationSummaryMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_core.documents import Document

# --- Configuration ---
# Set your OpenAI key: in Colab use the Secrets panel (key icon on left sidebar)
# os.environ["OPENAI_API_KEY"] = "sk-..."

PERSIST_DIR      = "./chroma_capstone"   # Chroma will write SQLite here
COLLECTION_NAME  = "capstone_docs"
CHUNK_SIZE       = 1000
CHUNK_OVERLAP    = 200
MMR_K            = 5
MMR_FETCH_K      = 20
EMBED_MODEL      = "text-embedding-3-small"
CHAT_MODEL       = "gpt-4o-mini"

# Create persist directory if it doesn't exist
Path(PERSIST_DIR).mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  Chroma persist dir : {PERSIST_DIR}")
print(f"  Chat model         : {CHAT_MODEL}")
print(f"  Embedding model    : {EMBED_MODEL}")
print(f"  MMR k={MMR_K}, fetch_k={MMR_FETCH_K}")

### What just happened?

- **Centralised config** means one place to tune — change `MMR_K` or `CHUNK_SIZE` and every
  downstream cell picks it up automatically.
- **`Path(PERSIST_DIR).mkdir(parents=True, exist_ok=True)`** is idempotent — safe to run
  multiple times; won't error if the directory already exists.
- **`text-embedding-3-small`** is the right default: ~5× cheaper than `ada-002` with better
  accuracy. For production, prefer `text-embedding-3-large` for highest quality.
- **All imports are at the top** of the first cell that uses them — this is the LangChain
  code-quality rule from the spec.


## Step 2 · Ingest PDFs and web pages

The ingestion layer uses two loaders:

| Loader | Input | Use when |
|--------|-------|----------|
| `PyPDFLoader` | Local `.pdf` path or URL to a PDF | Research papers, manuals, reports |
| `WebBaseLoader` | HTTP(S) URL | Documentation sites, blog posts, wikis |

Both return a list of `Document` objects with `page_content` (text) and `metadata`
(source, page number for PDFs).

**Strategy for diverse sources:** add at least one PDF and two web pages so the retriever
must synthesise across document types — a realistic production scenario.

> **Note:** For the capstone we use LangChain's own documentation pages as web sources and
> download a public PDF — both are free, stable, and relevant to the course content.


In [ ]:
import urllib.request

# ── Web sources (3 pages from LangChain docs) ───────────────────────────────
web_urls = [
    "https://python.langchain.com/docs/concepts/lcel/",
    "https://python.langchain.com/docs/concepts/retrievers/",
    "https://python.langchain.com/docs/concepts/memory/",
]

web_loader = WebBaseLoader(web_urls)
web_docs = web_loader.load()
print(f"Loaded {len(web_docs)} web documents")
for d in web_docs:
    print(f"  [{d.metadata.get('source', 'web')}] {len(d.page_content)} chars")

# ── PDF source — LangChain paper (public arxiv PDF) ─────────────────────────
PDF_URL = "https://arxiv.org/pdf/2302.00093"  # LangChain paper
pdf_path = "/tmp/langchain_paper.pdf"

print(f"\nDownloading PDF from {PDF_URL} ...")
urllib.request.urlretrieve(PDF_URL, pdf_path)

pdf_loader = PyPDFLoader(pdf_path)
pdf_docs = pdf_loader.load()
print(f"Loaded {len(pdf_docs)} PDF pages")
print(f"  First page preview: {pdf_docs[0].page_content[:200].strip()}")

# ── Combine all documents ────────────────────────────────────────────────────
all_docs = web_docs + pdf_docs
print(f"\nTotal documents loaded: {len(all_docs)}")

### What just happened?

- **`WebBaseLoader`** uses `requests` + `BeautifulSoup` under the hood to extract readable
  text from HTML pages — it strips nav, scripts, and footers automatically.
- **`PyPDFLoader`** extracts text page-by-page; each page becomes a separate `Document`
  with `metadata={"page": N, "source": filepath}`.
- **Multi-source ingestion** means answers can cite different document types in the same
  response — the retriever doesn't distinguish PDF vs. web; it only sees embeddings.
- **Combining lists** (`web_docs + pdf_docs`) is the correct pattern — both are `List[Document]`.


## Step 3 · Split documents and build the persistent Chroma store

`RecursiveCharacterTextSplitter` splits on `\n\n`, then `\n`, then ` ` — it respects
paragraph boundaries before falling back to sentence, then word boundaries.

**Why `chunk_overlap`?** Facts often straddle chunk boundaries. Overlap ensures the
sentence *"LangChain uses LCEL which is..."* appears in both the ending chunk and
the starting chunk of the next segment — so retrieval can find it regardless of
which chunk is matched.

**Persistent Chroma:** passing `persist_directory` to `Chroma` tells it to write a
SQLite file instead of keeping everything in memory. On subsequent runs you can load
the existing collection instead of re-embedding.

| Parameter | Value | Reasoning |
|-----------|-------|------------|
| `chunk_size=1000` | ~250 tokens | Fits 4-5 chunks in a 4K context window |
| `chunk_overlap=200` | 20% | Reduces boundary split at low cost |
| `add_start_index=True` | — | Adds character offset to metadata for debugging |


In [ ]:
# ── Split documents ──────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,  # useful for debugging which part of a doc was retrieved
)

splits = splitter.split_documents(all_docs)
print(f"Split {len(all_docs)} documents into {len(splits)} chunks")
print(f"Average chunk length: {sum(len(s.page_content) for s in splits) // len(splits)} chars")

# Show metadata on first chunk
print(f"\nSample chunk metadata: {splits[0].metadata}")
print(f"Sample chunk content (first 300 chars):\n{splits[0].page_content[:300]}")

# ── Build or reload persistent Chroma store ──────────────────────────────────
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# Check if the collection already exists to avoid re-embedding on restart
chroma_db_file = Path(PERSIST_DIR) / "chroma.sqlite3"

if chroma_db_file.exists():
    print("\nLoading existing Chroma collection (no re-embedding needed) ...")
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=PERSIST_DIR,
    )
    print(f"Collection loaded: {vectorstore._collection.count()} vectors")
else:
    print("\nBuilding Chroma collection (embedding all chunks) ...")
    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=PERSIST_DIR,  # writes to disk immediately
    )
    print(f"Embedded and stored {vectorstore._collection.count()} vectors")
    print(f"Chroma data written to: {PERSIST_DIR}")

### What just happened?

- **`add_start_index=True`** attaches `start_index` (character offset) to each chunk's
  metadata — invaluable for debugging when you need to find a chunk back in the source doc.
- **The persistence check** (`chroma_db_file.exists()`) prevents re-embedding on every
  run — embedding 500 chunks at each kernel restart would cost ~$0.01 and 30 seconds;
  at scale, skipping re-embedding is the difference between 10s and 10-minute startup.
- **`Chroma.from_documents(persist_directory=...)`** writes to disk in the same call that
  creates the collection — no separate `.persist()` call needed in recent Chroma versions.
- **`vectorstore._collection.count()`** gives the total number of stored vectors — a quick
  sanity check that ingestion completed.


## Step 4 · Build the MMR retriever

MMR (Maximal Marginal Relevance) re-ranks retrieved documents to balance:
- **Relevance** — how similar is this chunk to the query?
- **Diversity** — how different is this chunk from chunks already selected?

This prevents the retriever from returning 5 near-identical chunks from the same
paragraph, which would waste context window space without adding new information.

**Key parameters:**

| Parameter | Value | Effect |
|-----------|-------|--------|
| `k=5` | Return 5 chunks | Balances context richness vs. prompt length |
| `fetch_k=20` | Retrieve 20 candidates first | More candidates → better MMR selection |
| `lambda_mult=0.5` | 50/50 relevance vs. diversity | `1.0`=pure similarity, `0.0`=pure diversity |

> **Rule of thumb:** `fetch_k` should be 3–5× `k`. Fetching too few candidates defeats
> the purpose of MMR; fetching too many adds latency.


In [ ]:
# Build the MMR retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": MMR_K,             # number of chunks to return
        "fetch_k": MMR_FETCH_K, # candidates to consider before re-ranking
        "lambda_mult": 0.5,     # balance: 0=diverse, 1=similar
    },
)

# Verify retriever works and returns diverse results
test_query = "How does LangChain handle memory in conversational applications?"
retrieved_docs = retriever.invoke(test_query)

print(f"MMR retriever returned {len(retrieved_docs)} chunks for test query")
print(f"Query: {test_query}\n")

for i, doc in enumerate(retrieved_docs):
    source = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page", "-")
    preview = doc.page_content[:100].replace("\n", " ")
    print(f"[{i+1}] source={source} | page={page}")
    print(f"     {preview}...\n")

### What just happened?

- **`as_retriever(search_type="mmr")`** wraps the vector store with MMR logic — the
  underlying similarity search still happens in Chroma, but results are re-ranked before
  being returned.
- **Diversity in action:** you should see chunks from different source documents in the
  results — not 5 chunks from the same memory documentation page.
- **`lambda_mult=0.5`** is a starting point — increase toward 1.0 if your answers lack
  relevance; decrease toward 0.0 if you see too many near-duplicate chunks.
- **The retriever is a `Runnable`** — you can pipe it into LCEL chains or pass it directly
  to `ConversationalRetrievalChain`.


## Step 5 · Build the conversational chain with memory

`ConversationalRetrievalChain` handles the full conversational RAG loop:

```
user question + chat history
  → condense to standalone question (LLM call)
  → retrieve relevant chunks
  → generate answer with context + history
```

**Why `ConversationSummaryMemory`?** Buffer memory stores all messages verbatim — fine
for short sessions, but at 20+ turns you'll overflow the context window. Summary memory
uses a separate LLM call to compress the history into a running summary, keeping the
conversation bounded regardless of length.

| Memory class | Storage | Token cost | Context window safe? |
|--------------|---------|------------|---------------------|
| `ConversationBufferMemory` | All messages verbatim | O(n) | Only for short sessions |
| `ConversationBufferWindowMemory` | Last K turns | O(K) | Yes, but loses old context |
| `ConversationSummaryMemory` | LLM-generated summary | O(1) amortised | Yes — grows slowly |
| `ConversationSummaryBufferMemory` | Summary + recent buffer | O(K) | Best of both |


In [ ]:
# Initialise the LLM
llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0,
    streaming=True,  # enables .stream() on the chain
)

# ConversationSummaryMemory — compresses history into a running LLM summary
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",  # must match the key ConversationalRetrievalChain expects
    return_messages=True,       # returns messages, not a string — required for chat models
    output_key="answer",        # tell memory which output field to store
)

# ConversationalRetrievalChain wires retriever + memory + LLM together
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,  # include source docs in the response dict
    verbose=False,
)

print("ConversationalRetrievalChain built successfully.")
print(f"  LLM          : {CHAT_MODEL} (streaming=True)")
print(f"  Memory       : ConversationSummaryMemory")
print(f"  Retriever    : MMR k={MMR_K}")
print(f"  Source docs  : returned in response")

### What just happened?

- **`streaming=True`** on `ChatOpenAI` enables token-by-token output via `.stream()` —
  without this flag, the model waits for the full response before returning anything.
- **`return_messages=True`** in memory tells the chain to keep chat history as a list of
  `HumanMessage`/`AIMessage` objects — required for `ChatOpenAI` (vs. string-based models).
- **`output_key="answer"`** disambiguates which key in the chain's output dict should be
  written to memory — `ConversationalRetrievalChain` returns both `answer` and
  `source_documents`, so we must be explicit.
- **`return_source_documents=True`** lets the Gradio UI display citations alongside the answer.


## Step 6 · Test the chain with follow-up questions (memory health check)

The canonical memory health check is a series of follow-up questions that require
knowledge of prior turns:

1. Ask a factual question → get a grounded answer
2. Ask "What did you just say about X?" → memory should recall the answer
3. Ask a follow-up that uses a pronoun → memory should resolve the referent

If the chain treats every question as a fresh start, memory is broken.


In [ ]:
# Memory health check — 5 follow-up questions

def ask(question: str) -> dict:
    """Invoke the chain and return the result dict with answer + source_documents."""
    result = qa_chain.invoke({"question": question})
    return result

def print_answer(turn: int, question: str, result: dict) -> None:
    """Print a formatted turn summary."""
    answer = result["answer"]
    sources = result.get("source_documents", [])
    source_names = list({
        doc.metadata.get("source", "unknown") for doc in sources
    })
    print(f"\n{'='*60}")
    print(f"Turn {turn}: {question}")
    print(f"{'-'*60}")
    print(f"Answer: {answer[:300]}")
    print(f"Sources: {source_names}")

# Turn 1 — factual question about LCEL
r1 = ask("What is LCEL and what operators does it provide?")
print_answer(1, "What is LCEL and what operators does it provide?", r1)

# Turn 2 — memory recall: refers back to what was just said
r2 = ask("What did you just say about the pipe operator?")
print_answer(2, "What did you just say about the pipe operator?", r2)

# Turn 3 — pronoun resolution: 'it' refers to LCEL from turn 1
r3 = ask("Can it handle async calls too?")
print_answer(3, "Can it handle async calls too?", r3)

# Turn 4 — topic switch
r4 = ask("How does MMR retrieval work in LangChain?")
print_answer(4, "How does MMR retrieval work in LangChain?", r4)

# Turn 5 — connect both topics (LCEL + MMR) requiring memory of earlier turns
r5 = ask("Can I combine what you told me about LCEL and MMR into one retrieval chain?")
print_answer(5, "Can I combine LCEL and MMR into one retrieval chain?", r5)

# Print the memory summary to verify it's tracking conversation history
print("\n" + "="*60)
print("MEMORY SUMMARY (what ConversationSummaryMemory has stored):")
print("-"*60)
print(memory.moving_summary_buffer or "(empty — no summary generated yet)")

### What just happened?

- **Turn 2 (`"What did you just say about..."`)** is the canonical memory health check — if
  the chain returns a coherent recap of turn 1 without re-querying the vector store, memory
  is working correctly.
- **Turn 3 (`"Can it..."`)** tests pronoun resolution — the condensed question step uses
  memory to expand `"it"` into `"LCEL"` before hitting the retriever.
- **`memory.moving_summary_buffer`** shows the running LLM-generated summary — you should
  see content from turns 1–5 compressed into a few sentences.
- **`source_documents`** in the result dict shows which chunks were used — this feeds the
  Gradio UI's citation display in the next step.


## Step 7 · Add streaming output

`ConversationalRetrievalChain` doesn't natively expose `.stream()` at the chain level in
all LangChain versions. The reliable streaming pattern uses a callback handler to intercept
tokens as they arrive from the LLM, then yield them to the caller.

```
user question
  → chain.invoke()  (runs condense + retrieval synchronously)
  → LLM generates tokens → StreamingStdOutCallbackHandler captures each token
  → Gradio receives token stream via generator
```

For the Gradio UI, we'll wrap the chain in a generator function that accumulates
the streamed tokens and yields partial strings — this is what makes Gradio show
text appearing word-by-word.


In [ ]:
from threading import Thread
from queue import Queue, Empty
from langchain.callbacks.base import BaseCallbackHandler

class QueueCallbackHandler(BaseCallbackHandler):
    """Routes LLM tokens into a Queue so a generator can yield them."""

    def __init__(self, queue: Queue):
        self.queue = queue

    def on_llm_new_token(self, token: str, **kwargs) -> None:
        """Called by LangChain for every new token from the LLM."""
        self.queue.put(token)

    def on_llm_end(self, *args, **kwargs) -> None:
        """Signal that the LLM has finished generating."""
        self.queue.put(None)  # sentinel value — consumer stops when it sees None


def stream_response(question: str, chat_history: list) -> tuple[str, list]:
    """
    Run the chain with streaming and return (full_answer, source_docs).
    Yields partial answer strings so the caller can update a UI incrementally.
    """
    token_queue: Queue = Queue()
    handler = QueueCallbackHandler(token_queue)

    # Build a streaming-enabled LLM for this call
    streaming_llm = ChatOpenAI(
        model=CHAT_MODEL,
        temperature=0,
        streaming=True,
        callbacks=[handler],
    )

    # Build a fresh chain with the streaming LLM but the same retriever + memory
    streaming_chain = ConversationalRetrievalChain.from_llm(
        llm=streaming_llm,
        retriever=retriever,
        memory=memory,
        return_source_documents=True,
        verbose=False,
    )

    result_container = {}

    def run_chain():
        result_container["result"] = streaming_chain.invoke({"question": question})

    # Run the chain in a background thread so we can read tokens concurrently
    thread = Thread(target=run_chain, daemon=True)
    thread.start()

    # Collect tokens as they arrive
    full_answer = ""
    while True:
        try:
            token = token_queue.get(timeout=30)  # 30-second safety timeout
            if token is None:  # LLM signalled completion
                break
            full_answer += token
        except Empty:
            break  # timeout — stop waiting

    thread.join(timeout=5)
    sources = result_container.get("result", {}).get("source_documents", [])
    return full_answer, sources


# Quick streaming test
print("Testing streaming output...")
test_answer, test_sources = stream_response(
    "What is LCEL in one sentence?",
    chat_history=[],
)
print(f"Streamed answer: {test_answer}")
print(f"Source count: {len(test_sources)}")

### What just happened?

- **`QueueCallbackHandler`** decouples the token producer (LLM thread) from the consumer
  (Gradio's generator) using a thread-safe `Queue` — this is the standard pattern when you
  need streaming output from a synchronous LangChain chain.
- **Background thread** runs the chain so the main thread can read from the queue without
  blocking — the generator yields partial strings while the LLM is still generating.
- **Sentinel `None`** signals end-of-stream — a common pattern for producer/consumer queues
  in Python; the consumer breaks out of its loop cleanly without polling.
- **`timeout=30`** on `queue.get()` is a safety valve — if the LLM hangs, the generator
  exits rather than blocking forever.


## Step 8 · Build the Gradio ChatInterface

Gradio's `ChatInterface` expects a function with the signature:

```python
def chat_fn(message: str, history: list[list[str]]) -> str | Generator[str, None, None]:
    ...
```

- `message` — the latest user message
- `history` — list of `[user_msg, bot_msg]` pairs from previous turns
- Return a **generator** of partial strings for streaming; return a plain string for blocking

We also add a **sources accordion** that expands below each response to show which
documents were cited — critical for production RAG transparency.

| Gradio component | Purpose |
|------------------|---------|
| `ChatInterface` | Main chat UI — handles history display automatically |
| `Accordion` | Collapsible container for source citations |
| `Textbox` | Shows formatted source metadata below each answer |

Reference: https://www.gradio.app/docs/gradio/chatinterface


In [ ]:
import gradio as gr

# Shared state for source tracking between chat_fn and the sources display
_last_sources: list = []


def format_sources(source_docs: list) -> str:
    """Format source documents into a human-readable citation list."""
    if not source_docs:
        return "No sources retrieved."
    lines = []
    seen = set()
    for i, doc in enumerate(source_docs):
        src = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", None)
        key = (src, page)
        if key in seen:
            continue
        seen.add(key)
        loc = f" (page {page + 1})" if page is not None else ""
        preview = doc.page_content[:120].replace("\n", " ")
        lines.append(f"[{len(lines)+1}] {src}{loc}\n    ...{preview}...")
    return "\n\n".join(lines)


def chat_fn(message: str, history: list) -> str:
    """
    Gradio ChatInterface handler.
    Returns the full answer as a string; sources are stored in _last_sources.
    """
    global _last_sources

    if not message.strip():
        return "Please enter a question."

    full_answer, sources = stream_response(message, history)
    _last_sources = sources
    return full_answer


def get_sources_display() -> str:
    """Return formatted sources from the most recent response."""
    return format_sources(_last_sources)


def reset_memory() -> tuple[list, str]:
    """Clear ConversationSummaryMemory and reset the sources display."""
    memory.clear()  # wipes chat_history and moving_summary_buffer
    return [], "Memory cleared — starting a new conversation."


# Build the Gradio interface
with gr.Blocks(title="RAG Chatbot — LLM Engineering Capstone") as demo:
    gr.Markdown("""
    # 📚 RAG Chatbot — LLM Engineering Capstone
    Multi-document chatbot powered by LangChain + Chroma + GPT-4o-mini.
    Ask questions about LCEL, retrievers, memory, and agents.
    """)

    chatbot = gr.ChatInterface(
        fn=chat_fn,
        title="",                          # title already set in Blocks
        retry_btn=None,
        undo_btn=None,
        clear_btn="🗑 Clear chat",
        examples=[
            "What is LCEL and why does it matter?",
            "How does MMR retrieval differ from regular similarity search?",
            "Which memory class should I use for a long multi-turn conversation?",
        ],
    )

    with gr.Accordion("📎 Sources from last answer", open=False):
        sources_box = gr.Textbox(
            label="Retrieved documents",
            lines=8,
            interactive=False,
        )
        refresh_btn = gr.Button("Refresh sources")
        refresh_btn.click(fn=get_sources_display, outputs=sources_box)

    with gr.Row():
        memory_btn = gr.Button("🔄 Reset memory", variant="secondary")
        memory_status = gr.Textbox(label="Memory status", interactive=False, scale=3)
    memory_btn.click(fn=reset_memory, outputs=[chatbot.chatbot, memory_status])


print("Gradio interface built. Call demo.launch() to start.")

### What just happened?

- **`gr.Blocks` + `gr.ChatInterface`** gives full layout control while keeping the
  chat UI component — `ChatInterface` alone doesn't support adding extra components below it.
- **`examples=[...]`** adds clickable starter questions to the chat UI — good UX for demos.
- **`Accordion` for sources** keeps the UI clean by default but lets users inspect citations
  — production RAG apps must show sources to be trustworthy.
- **`memory_btn.click(fn=reset_memory)`** clears `ConversationSummaryMemory` and resets the
  chat history in the UI simultaneously — the memory reset must happen on both sides.


In [ ]:
# Launch the Gradio interface
# share=True creates a temporary public URL — useful in Colab where localhost isn't accessible
demo.launch(share=True)

### What just happened?

- **`share=True`** generates a `gradio.live` URL valid for 72 hours — the only way to access
  a Gradio app running in Colab from your browser.
- **The app is live** — open the URL, ask questions from the examples list, then click
  "Refresh sources" to see which documents drove the answer.
- **Memory test:** ask the same question twice, then ask "What did you just say about X?"
  — the second answer should reference the first.
- **To stop the server:** run `demo.close()` or interrupt the kernel.


## Challenge — End-to-end capstone implementation

Implement the full production RAG chatbot from scratch in the cells below.
All pieces have been demonstrated above — your task is to wire them together.

**Requirements (each must be implemented — not just described):**

1. **Ingest** at least 3 documents: 1 PDF (any publicly accessible PDF URL) + 2 web pages
2. **Split** all documents with `RecursiveCharacterTextSplitter` (choose your own chunk_size)
3. **Store** embeddings in a persistent Chroma collection at a path you specify
4. **Build** an MMR retriever with `k=5` and `fetch_k` at least 3× k
5. **Create** a `ConversationSummaryMemory` and wire it into a `ConversationalRetrievalChain`
6. **Implement** streaming using the `QueueCallbackHandler` pattern (or `.stream()` if your
   LangChain version supports it natively on the chain)
7. **Build** a Gradio `ChatInterface` that:
   - Shows sources for each response
   - Has a "Reset memory" button that clears both the chain memory and the chat history
   - Includes at least 3 example questions in the interface
8. **Run the memory health check:** ask at least 5 follow-up questions where at least 2
   require knowledge of prior turns ("What did you just say about X?", "Can it also...?")

**Evaluation criteria:**
- Does the Gradio app launch and respond without errors?
- Does the "Reset memory" button clear the chain's memory state?
- Do follow-up questions that reference prior turns get coherent answers?
- Are sources shown and non-empty for substantive questions?


In [ ]:
# Challenge: Full capstone implementation from scratch

# ── 1. Imports ───────────────────────────────────────────────────────────────
# import ...

# ── 2. Configuration ─────────────────────────────────────────────────────────
# MY_PERSIST_DIR = "./my_chroma_capstone"
# MY_PDF_URL = "..."
# MY_WEB_URLS = [...]

# ── 3. Ingest: PDF + web pages ───────────────────────────────────────────────
# pdf_loader = PyPDFLoader(...)
# web_loader = WebBaseLoader(...)
# all_docs = ...

# ── 4. Split ─────────────────────────────────────────────────────────────────
# splitter = RecursiveCharacterTextSplitter(...)
# splits = splitter.split_documents(all_docs)

# ── 5. Persistent Chroma store ───────────────────────────────────────────────
# my_vectorstore = Chroma.from_documents(..., persist_directory=MY_PERSIST_DIR)

# ── 6. MMR retriever ─────────────────────────────────────────────────────────
# my_retriever = my_vectorstore.as_retriever(search_type="mmr", search_kwargs={...})

# ── 7. Memory + chain ────────────────────────────────────────────────────────
# my_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, streaming=True)
# my_memory = ConversationSummaryMemory(...)
# my_chain = ConversationalRetrievalChain.from_llm(...)

# ── 8. Streaming handler ─────────────────────────────────────────────────────
# class MyQueueHandler(BaseCallbackHandler):
#     ...

# def my_stream_response(question, history):
#     ...

# ── 9. Gradio ChatInterface with sources + memory reset ──────────────────────
# with gr.Blocks(title="My RAG Chatbot") as my_demo:
#     ...

# ── 10. Memory health check (5 follow-up questions) ──────────────────────────
# questions = [
#     "...",               # factual
#     "What did you just say about ...?",  # memory recall
#     "Can it also ...?",  # pronoun resolution
#     "...",               # topic switch
#     "How do those two concepts relate?",  # synthesis
# ]
# for q in questions:
#     result = my_chain.invoke({"question": q})
#     print(f"Q: {q}\nA: {result['answer'][:200]}\n")

# ── 11. Launch ───────────────────────────────────────────────────────────────
# my_demo.launch(share=True)

---
## Day 14 key concepts recap

| Concept | What to remember |
|---------|------------------|
| `PyPDFLoader` | One `Document` per PDF page; `metadata["page"]` is 0-indexed |
| `WebBaseLoader` | Strips HTML; pass a list of URLs; one doc per URL |
| `RecursiveCharacterTextSplitter` | Splits on `\n\n` → `\n` → ` `; `add_start_index=True` for debugging |
| Persistent Chroma | Pass `persist_directory` to `from_documents` — no separate `.persist()` needed |
| Reload existing collection | Use `Chroma(collection_name=..., persist_directory=...)` to skip re-embedding |
| MMR retriever | `search_type="mmr"`, `fetch_k` ≥ 3× `k`, `lambda_mult=0.5` |
| `ConversationSummaryMemory` | `return_messages=True`, `output_key="answer"` — both required for chat models |
| `return_source_documents=True` | Adds `source_documents` key to chain output — needed for citations |
| `QueueCallbackHandler` | Thread-safe token streaming: producer (LLM thread) → Queue → consumer (Gradio) |
| `gr.Blocks` + `gr.ChatInterface` | Combine for layout control + built-in chat history management |
| Memory health check | `"What did you just say about X?"` is the canonical memory verification query |

> **Tip:** Test with at least 5 follow-up questions that require memory — 'What did you just say about X?' is the canonical memory health check.

---
## Congratulations — you've completed the LLM Engineering with LangChain course!

You have built a **production-ready multi-document RAG chatbot** that:
- Ingests PDFs and web pages with `PyPDFLoader` and `WebBaseLoader`
- Stores embeddings persistently in Chroma
- Retrieves diverse, relevant context with MMR re-ranking
- Remembers long conversations with `ConversationSummaryMemory`
- Streams responses token-by-token
- Exposes the full pipeline through a Gradio interface with source citations

Mark Day 14 complete in your [tracker](../index.html).
